<a href="https://colab.research.google.com/github/trefftzc/voronoiWithJulia/blob/main/Exploring_JACC_GPU.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Exploring JACC using a GPU
ACC is a library that makes it possible to write parallel code that will run on either GPUs or microprocessors with several cores, withouth having to change the code.

One selects a backend to decide where the code will be executed, either on a CPU with several cores or on a GPU.

In this notebook, the code runs on a GPU.

In [1]:
using Pkg

In [2]:
Pkg.add("JACC")

   Resolving package versions...
     Project No packages added to or removed from `~/.julia/environments/v1.12/Project.toml`
    Manifest No packages added to or removed from `~/.julia/environments/v1.12/Manifest.toml`
Precompiling packages...
   3811.3 ms  ✓ JACC
   9069.1 ms  ✓ JACC → CUDAExt
  2 dependencies successfully precompiled in 14 seconds. 532 already precompiled.


In [3]:
import JACC

In [4]:
JACC.set_backend("CUDA")

┌ Info: Backend preferences deleted
└ Restart your Julia session for this change to take effect!
┌ Info: Added "cuda" backend
└ Restart your Julia session for this change to take effect!
┌ Info: New default backend set: "cuda"
└ Restart your Julia session for this change to take effect!


In [4]:
JACC.@init_backend

[ Info: CUDA backend loaded


In [5]:
using JACC
import JACC

# JACC.@init_backend

# Kernel to find the closest seed

function distance(seedX::Int64,seedY::Int64,x::Int64,y::Int64)
    diff_x = seedX - x
    diff_y = seedY - y
    sum_squares = diff_x * diff_x + diff_y * diff_y
    result = sqrt(sum_squares)
    res::Int64 = round(Int64, result)
    return res
end

function closestSeed(x,y,nSeeds,seedsX::Vector{Int64},seedsY::Vector{Int64})
    #::Array(::Core.Int8)seedsX,::Array(::Core.Int8)seedsY)
    closest_seed = -1
    closest_dist = typemax(Int64)
    for seed =1:nSeeds
        dist = distance(seedsX[seed],seedsY[seed],x,y)
        if dist < closest_dist
            closest_dist = dist
            closest_seed = seed
        end
    end
    return closest_seed
end

function closestSeedDevice(x,y,canvasDevice,nSeeds,seedsX,seedsY)
    #::Array(::Core.Int8)seedsX,::Array(::Core.Int8)seedsY)
    closest_seed = -1
    closest_dist = typemax(Int64)
    for seed =1:nSeeds
        dist = distance(seedsX[seed],seedsY[seed],x,y)
        if dist < closest_dist
            closest_dist = dist
            closest_seed = seed
        end
    end
    canvasDevice[x,y] = closest_seed
end
JACC.@init_backend
nSeeds = 4
sizeCanvas = 64
canvas = zeros(Core.Int64,(sizeCanvas,sizeCanvas))
seedsX = zeros(Core.Int64,nSeeds)
seedsY = zeros(Core.Int64,nSeeds)
seedsX[1] = 1
seedsX[2] = 64
seedsX[3] = 1
seedsX[4] = 64
seedsY[1] = 1
seedsY[2] = 1
seedsY[3] = 64
seedsY[4] = 64
canvas_dev = JACC.to_device(canvas)
seedsX_dev = JACC.to_device(seedsX)
seedsY_dev = JACC.to_device(seedsY)

# Sequential version
println("Output of the sequential version:\n")
@inbounds begin
  for y = 1:sizeCanvas
    for x = 1:sizeCanvas
        canvas[x,y] = closestSeed(x,y,nSeeds,seedsX,seedsY)
    end
  end
end


# Print the results

for x = 1:sizeCanvas
    for y = 1:sizeCanvas
        print(canvas[x,y] )
    end
    println()
end
# Now the parallel version

JACC.@parallel_for range=(sizeCanvas,sizeCanvas) closestSeedDevice(canvas_dev,nSeeds, seedsX_dev, seedsY_dev) # cannot use spaces for keyword arguments

canvas2 = JACC.to_host(canvas_dev)
println("Output of the parallel version:\n")
for x = 1:sizeCanvas
    for y = 1:sizeCanvas
        print(canvas2[x,y] )
    end
    println()
end
println("Done")

Output of the sequential version:



[ Info: CUDA backend loaded


1111111111111111111111111111111133333333333333333333333333333333
1111111111111111111111111111111133333333333333333333333333333333
1111111111111111111111111111111133333333333333333333333333333333
1111111111111111111111111111111133333333333333333333333333333333
1111111111111111111111111111111133333333333333333333333333333333
1111111111111111111111111111111133333333333333333333333333333333
1111111111111111111111111111111133333333333333333333333333333333
1111111111111111111111111111111133333333333333333333333333333333
1111111111111111111111111111111133333333333333333333333333333333
1111111111111111111111111111111133333333333333333333333333333333
1111111111111111111111111111111133333333333333333333333333333333
1111111111111111111111111111111133333333333333333333333333333333
1111111111111111111111111111111133333333333333333333333333333333
1111111111111111111111111111111133333333333333333333333333333333
1111111111111111111111111111111133333333333333333333333333333333
1111111111111111111111111

Now with a larger canvas.

In [15]:
# voronoi_with_jacc_size_4096

using JACC
import JACC

const SIZE_AREA_P = 4096
# JACC.@init_backend

# Kernel to find the closest seed

function distance(seedX::Int64,seedY::Int64,x::Int64,y::Int64)
    diff_x = seedX - x
    diff_y = seedY - y
    sum_squares = diff_x * diff_x + diff_y * diff_y
    result = sqrt(sum_squares)
    res::Int64 = round(Int64, result)
    return res
end

function closestSeed(x,y,nSeeds,seedsX::Vector{Int64},seedsY::Vector{Int64})
    #::Array(::Core.Int8)seedsX,::Array(::Core.Int8)seedsY)
    closest_seed = -1
    closest_dist = typemax(Int64)
    for seed =1:nSeeds
        dist = distance(seedsX[seed],seedsY[seed],x,y)
        if dist < closest_dist
            closest_dist = dist
            closest_seed = seed
        end
    end
    return closest_seed
end

function closestSeedDevice(x,y,canvasDevice,nSeeds,seedsX,seedsY)
    #::Array(::Core.Int8)seedsX,::Array(::Core.Int8)seedsY)
    closest_seed = -1
    closest_dist = typemax(Int64)
    for seed =1:nSeeds
        dist = distance(seedsX[seed],seedsY[seed],x,y)
        if dist < closest_dist
            closest_dist = dist
            closest_seed = seed
        end
    end
    canvasDevice[x,y] = closest_seed
end

JACC.@init_backend
nSeeds = 4

println("SIZE_AREA_P is: ",SIZE_AREA_P)
canvas = zeros(Core.Int64,(SIZE_AREA_P,SIZE_AREA_P))
canvas2 = zeros(Core.Int64,(SIZE_AREA_P,SIZE_AREA_P))
seedsX = zeros(Core.Int64,nSeeds)
seedsY = zeros(Core.Int64,nSeeds)
seedsX[1] = 1
seedsX[2] = SIZE_AREA_P
seedsX[3] = 1
seedsX[4] = SIZE_AREA_P
seedsY[1] = 1
seedsY[2] = 1
seedsY[3] = SIZE_AREA_P
seedsY[4] = SIZE_AREA_P
canvas_dev = JACC.to_device(canvas)
seedsX_dev = JACC.to_device(seedsX)
seedsY_dev = JACC.to_device(seedsY)
# Sequential version
@time @sync begin
  @inbounds begin
    for y = 1:SIZE_AREA_P
        for x = 1:SIZE_AREA_P
            canvas[x,y] = closestSeed(x,y,nSeeds,seedsX,seedsY)
        end
    end
  end
end
println("Sequential time after compilation: ")
@time @sync begin
  @inbounds begin
    for y = 1:SIZE_AREA_P
        for x = 1:SIZE_AREA_P
            canvas[x,y] = closestSeed(x,y,nSeeds,seedsX,seedsY)
        end
    end
  end
end

# Now the parallel version
println("Compiling the parallel version: ")
@time @sync begin
    JACC.@parallel_for range=(SIZE_AREA_P,SIZE_AREA_P) closestSeedDevice(canvas_dev,nSeeds, seedsX_dev, seedsY_dev) # cannot use spaces for keyword arguments
end
@time @sync begin
  canvas2 = JACC.to_host(canvas_dev)
end
println("Now running the version that has been compiled already: ")
@time @sync begin
    JACC.@parallel_for range=(SIZE_AREA_P,SIZE_AREA_P) closestSeedDevice(canvas_dev,nSeeds, seedsX_dev, seedsY_dev) # cannot use spaces for keyword arguments
end
@time @sync begin
  canvas2 = JACC.to_host(canvas_dev)
end

# Compare the results
#outputs_match = true
#for y = 1:SIZE_AREA_P
#    for x = 1:SIZE_AREA_P
#        if canvas[x,y] != canvas2[x,y]
#            println("Mismatch at (y): canvas = (canvas2[x,y])")
#            global
#            outputs_match = false
#        end
#    end
#end

#if outputs_match
#    println("Outputs match!")
#else
#    println("Outputs do not match.")
#end

SIZE_AREA_P is: 4096
  2.881798 seconds (58.75 M allocations: 896.760 MiB, 21.16% gc time, 0.76% compilation time)


[ Info: CUDA backend loaded


Sequential time after compilation: 
  1.851087 seconds (58.74 M allocations: 896.250 MiB, 3.39% gc time)
Compiling the parallel version: 
  0.252479 seconds (88.78 k allocations: 6.453 MiB, 1.70% compilation time: <1% of which was recompilation)
  0.094101 seconds (25 allocations: 128.005 MiB, 7.01% gc time)
Now running the version that has been compiled already: 
  0.025994 seconds (333 allocations: 7.359 KiB)
  0.093208 seconds (17 allocations: 128.004 MiB, 10.45% gc time)


4096×4096 Matrix{Int64}:
 1  1  1  1  1  1  1  1  1  1  1  1  1  …  3  3  3  3  3  3  3  3  3  3  3  3
 1  1  1  1  1  1  1  1  1  1  1  1  1     3  3  3  3  3  3  3  3  3  3  3  3
 1  1  1  1  1  1  1  1  1  1  1  1  1     3  3  3  3  3  3  3  3  3  3  3  3
 1  1  1  1  1  1  1  1  1  1  1  1  1     3  3  3  3  3  3  3  3  3  3  3  3
 1  1  1  1  1  1  1  1  1  1  1  1  1     3  3  3  3  3  3  3  3  3  3  3  3
 1  1  1  1  1  1  1  1  1  1  1  1  1  …  3  3  3  3  3  3  3  3  3  3  3  3
 1  1  1  1  1  1  1  1  1  1  1  1  1     3  3  3  3  3  3  3  3  3  3  3  3
 1  1  1  1  1  1  1  1  1  1  1  1  1     3  3  3  3  3  3  3  3  3  3  3  3
 1  1  1  1  1  1  1  1  1  1  1  1  1     3  3  3  3  3  3  3  3  3  3  3  3
 1  1  1  1  1  1  1  1  1  1  1  1  1     3  3  3  3  3  3  3  3  3  3  3  3
 1  1  1  1  1  1  1  1  1  1  1  1  1  …  3  3  3  3  3  3  3  3  3  3  3  3
 1  1  1  1  1  1  1  1  1  1  1  1  1     3  3  3  3  3  3  3  3  3  3  3  3
 1  1  1  1  1  1  1  1  1  1  1  1  1 